# 02 - De RDDs a DataFrames y Catalyst Optimizer

### Antipatrón de RDDs en Python
Los RDDs sufren sobrecarga de serialización entre la JVM y Python (Py4J). Los DataFrames usan memoria binaria fuera del recolector de basura (Project Tungsten) y optimizan consultas automáticamente mediante el motor **Catalyst**.


In [ ]:
import sys
sys.path.append("..")
from src.config import get_spark_session
from src.etl.olympics_pipeline import DEPORTISTAS_SCHEMA
import pyspark.sql.functions as F

spark = get_spark_session("02_DataFrames")
df = spark.read.schema(DEPORTISTAS_SCHEMA).option("header", "true").csv("../data/raw/deportista.csv")
df.printSchema()
df.show(5)


### Inspección del Plan Físico (`explain`)
Observa cómo Catalyst elimina columnas y filtra antes de cargar datos:


In [ ]:
df_plan = df.filter(F.col("edad") > 20).select("nombre", "peso")
df_plan.explain(mode="formatted")


### Ejercicio Práctico 2
Calcula la columna `peso_lb` multiplicando el peso por 2.20462 redondeado a 1 decimal.


In [ ]:
# Solución validada:
df_calc = df.filter(F.col("peso").isNotNull()).withColumn("peso_lb", F.round(F.col("peso") * 2.20462, 1))
df_calc.select("nombre", "peso", "peso_lb").show(5)
